# Debug: Claude Opus 4.7 Refusal

Submit a single benchmark query to `claude-opus-4-7` and inspect the full API response.

In [ ]:
import os, sys
sys.path.insert(0, '../src')

import anthropic
from dotenv import load_dotenv
from utils import parse_csv, build_query

load_dotenv(override=True)

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

In [ ]:
# Build the first benchmark query
configs = parse_csv('../docs/virseq_benchmark.csv')
query = build_query(configs[0], use_gget_virus=False, return_integer_only=True)
print(f'Query (query_id={configs[0].query_id}, {configs[0].pathogen}):')
print(query)

In [ ]:
from benchmark_claude import SYSTEM_PROMPT, EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL

print('System prompt:')
print(SYSTEM_PROMPT)

In [ ]:
# Send to Opus 4.7 with tools
MODEL = 'claude-opus-4-7'

response = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    system=SYSTEM_PROMPT,
    tools=[EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL],
    messages=[{'role': 'user', 'content': query}],
)

print(f'stop_reason: {response.stop_reason}')
print(f'model: {response.model}')
print(f'usage: {response.usage}')
print(f'content blocks: {len(response.content)}')
print()
for i, block in enumerate(response.content):
    print(f'--- Block {i} ---')
    print(f'type: {block.type}')
    print(block)
    print()

In [ ]:
# Try without tools to see if refusal is query-related or tool-related
response_no_tools = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': query}],
)

print(f'stop_reason: {response_no_tools.stop_reason}')
print(f'usage: {response_no_tools.usage}')
print()
for i, block in enumerate(response_no_tools.content):
    print(f'--- Block {i} ---')
    print(f'type: {block.type}')
    print(block)
    print()

In [ ]:
# Try with higher max_tokens in case thinking tokens are the issue
response_high = client.messages.create(
    model=MODEL,
    max_tokens=16384,
    system=SYSTEM_PROMPT,
    tools=[EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL],
    messages=[{'role': 'user', 'content': query}],
)

print(f'stop_reason: {response_high.stop_reason}')
print(f'usage: {response_high.usage}')
print()
for i, block in enumerate(response_high.content):
    print(f'--- Block {i} ---')
    print(f'type: {block.type}')
    print(block)
    print()